# 05 — Reverse stress as certified scenario design

Forward stress testing asks "given this scenario, how bad are losses?"
Reverse stress inverts it: "given a loss we cannot survive, what is the **most
plausible** scenario that causes it?" Two components:

1. **Sample-based design** — among generated paths, the most plausible one
   breaching each PD target (from the frontier's design table).
2. **A certificate** — the satellite is monotone, so the worst case over a
   plausibility box is a known corner: one forward pass, provable, no search.


In [1]:
import os, sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Find the folder that contains the `multistate` package (works from
# bl225msc/ or from a self-contained thesis_demo/ bundle).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "multistate").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if (ROOT / "data" / "fed").is_dir():
    os.environ.setdefault("THESIS_DATA_DIR", str(ROOT / "data"))
RESULTS = ROOT / "results"
print(f"root: {ROOT}")

def load_csv(path, produced_by):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}\n  produce it with:\n  {produced_by}")
        return None
    return pd.read_csv(path)

def load_pickle(path, produced_by):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}\n  produce it with:\n  {produced_by}")
        return None
    with path.open("rb") as fh:
        return pickle.load(fh)

root: c:\Users\user\OneDrive\Desktop\thesis_clean\bl225msc\thesis_demo


In [2]:
design = load_csv(RESULTS / "scengen" / "frontier_semantics3" / "design_table.csv",
                  "any frontier run writes design_table.csv")
if design is not None:
    display(design.round(4))
    print("breach_share: fraction of generated paths exceeding each PD target;")
    print("best_plausibility_breaching: the least-surprising path that still breaches - the design output.")

,generator,pd_target,breach_share,best_plausibility_breaching,best_breaching_path
0,graph_gaussian,0.20,0.60,-217.6537,graph_gaussian_0045
1,graph_gaussian,0.25,0.32,-217.6537,graph_gaussian_0045
2,graph_gaussian,0.30,0.20,-217.6537,graph_gaussian_0045
3,graph_gaussian,0.40,0.08,-229.0956,graph_gaussian_0084
4,graph_gaussian_obs_u,0.20,0.00,NaN,NaN
5,graph_gaussian_obs_u,0.25,0.00,NaN,NaN
6,graph_gaussian_obs_u,0.30,0.00,NaN,NaN
7,graph_gaussian_obs_u,0.40,0.00,NaN,NaN
8,graph_gaussian_do_u,0.20,0.00,NaN,NaN
9,graph_gaussian_do_u,0.25,0.00,NaN,NaN


breach_share: fraction of generated paths exceeding each PD target;
best_plausibility_breaching: the least-surprising path that still breaches - the design output.


## The certificate cross-audit

Monotonicity implies **no path inside the certified box may exceed the corner
PD** (24.25% for the promoted GAM). Generated paths that beat the corner must
provably have left the box. Verified on the saved runs: 14/14 and 65/65
over-corner paths breached at least one box limit — the generated ensemble
independently confirms the certificate's boundary.

In [3]:
# Live re-audit against whichever frontier run is present
paths_csv = RESULTS / "scengen" / "frontier_semantics3" / "frontier_macro_paths.csv"
pds_csv = RESULTS / "scengen" / "frontier_semantics3" / "frontier_paths.csv"
CORNER = 0.2425  # certified box-corner H12 PD (promoted GAM)
BOX = {"unemployment": ("max", 6.0), "vix": ("max", 40.0),
       "mortgage_30y": ("max", 3.0), "hpi_qoq_growth": ("min", -15.0)}
if paths_csv.exists() and pds_csv.exists():
    macro = pd.read_csv(paths_csv); pds = pd.read_csv(pds_csv)
    from multistate.scengen.macro import FEATURES, load_macro_history
    jump = load_macro_history()[FEATURES].iloc[-1]
    over = pds[pds["pd_h12"] > CORNER]["scenario"]
    breached = 0
    for path_id in over:
        frame = macro[macro["path_id"].eq(path_id)]
        out = (frame["unemployment"].max() > jump["unemployment"] + BOX["unemployment"][1]
               or frame["vix"].max() > jump["vix"] + BOX["vix"][1]
               or frame["mortgage_30y"].max() > jump["mortgage_30y"] + BOX["mortgage_30y"][1]
               or frame["hpi_qoq_growth"].min() < BOX["hpi_qoq_growth"][1])
        breached += bool(out)
    print(f"paths over the {CORNER:.2%} corner: {len(over)}; provably outside the box: {breached}")
    print("certificate consistent" if breached == len(over) else "INVESTIGATE: an in-box path beat the corner")
else:
    print("frontier artifacts not found - run the frontier first")

paths over the 24.25% corner: 64; provably outside the box: 64
certificate consistent


**Takeaways.** Reverse stress becomes *design*: a plausibility-ranked list of
breaching scenarios a committee can actually discuss, plus a mathematical bound a
black-box model cannot offer (the RF satellite never even reaches a 15% target —
"false comfort"). This closes the loop the thesis opened: generators are graded,
scenarios are priced, semantics are measured, and the worst case is certified.